# 02 — Randomized Experiments
**References:** Fisher (1935) *The Design of Experiments* · Neyman (1923/1990) · Imbens & Rubin (2015) Ch. 5–7 · Lin (2013, *Annals of Applied Statistics*) · Stanford STATS 361 Lecture 2 · MIT 14.310x

## Narrative thread
```
Why RCTs are the gold standard -> Fisher's exact p-value -> Neyman's estimator & variance -> Covariate adjustment (Lin) -> Stratified designs
```

## Two modes of inference from the same experiment

**Fisher (randomization inference).** Test the *sharp null* $H_0: Y_i(1) = Y_i(0)$ for
**every** unit. Under the sharp null, all potential outcomes are known, so the distribution
of any test statistic over re-randomizations is known **exactly** — no models, no asymptotics.

**Neyman (repeated sampling).** Estimate the ATE with the difference in means
$$\hat{\tau} = \bar{Y}_1^{obs} - \bar{Y}_0^{obs}, \qquad
\widehat{\text{Var}}(\hat{\tau}) = \frac{s_1^2}{n_1} + \frac{s_0^2}{n_0}$$

Neyman's variance estimator is **conservative**: its expectation exceeds the true
randomization variance by $S_{\tau}^2 / n$ (the variance of individual effects), which is
unobservable and zero only under constant effects.

| | Fisher | Neyman |
|---|---|---|
| Null | Sharp: $\tau_i = 0$ for all $i$ | Average: $\bar{\tau} = 0$ |
| Output | Exact p-value | Estimate + CI |
| Assumptions | Randomization only | Randomization only (CLT for the CI) |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [ ]:
# ── Fisher randomization test: exact inference from randomization alone ──
# Small experiment: 6 treated, 6 control
y_t = np.array([31, 28, 35, 27, 30, 33])
y_c = np.array([25, 26, 29, 24, 28, 22])
y_all = np.concatenate([y_t, y_c])
n_t = len(y_t)
obs_stat = y_t.mean() - y_c.mean()

# Enumerate ALL possible assignments of 6 treated among 12 units
from itertools import combinations
null_stats = []
for treat_idx in combinations(range(len(y_all)), n_t):
    mask = np.zeros(len(y_all), bool)
    mask[list(treat_idx)] = True
    # Under the sharp null, outcomes are fixed regardless of assignment
    null_stats.append(y_all[mask].mean() - y_all[~mask].mean())
null_stats = np.array(null_stats)

p_exact = (np.abs(null_stats) >= np.abs(obs_stat) - 1e-12).mean()
print(f'Observed difference in means: {obs_stat:.3f}')
print(f'Number of possible assignments: {len(null_stats)}  (C(12,6) = 924)')
print(f'Exact two-sided randomization p-value: {p_exact:.4f}')
print(f't-test p-value (for comparison):       {stats.ttest_ind(y_t, y_c).pvalue:.4f}')

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(null_stats, bins=40, color='#1f77b4', alpha=0.8)
ax.axvline(obs_stat, color='#d62728', lw=2, label=f'observed = {obs_stat:.2f}')
ax.axvline(-obs_stat, color='#d62728', lw=2, ls='--')
ax.set_title('Exact randomization distribution under the sharp null')
ax.set_xlabel('difference in means'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Neyman: difference in means, and why its variance is conservative ────
# Simulate a finite population with HETEROGENEOUS effects, re-randomize many times
n = 100
y0 = np.random.normal(20, 5, n)
tau_i = np.random.normal(4, 6, n)          # individual effects vary a lot
y1 = y0 + tau_i
true_ate = tau_i.mean()

n_t = 50
reps = 20_000
est, var_hat = [], []
for _ in range(reps):
    w = np.zeros(n, int)
    w[np.random.choice(n, n_t, replace=False)] = 1
    yo = np.where(w == 1, y1, y0)
    d = yo[w == 1].mean() - yo[w == 0].mean()
    v = yo[w == 1].var(ddof=1)/n_t + yo[w == 0].var(ddof=1)/(n - n_t)
    est.append(d); var_hat.append(v)
est, var_hat = np.array(est), np.array(var_hat)

print(f'True (finite-population) ATE:        {true_ate:.3f}')
print(f'Mean of estimates over re-randomizations: {est.mean():.3f}  -> unbiased')
print(f'True randomization variance:  {est.var():.4f}')
print(f'Mean Neyman variance estimate: {var_hat.mean():.4f}  -> conservative (larger)')
print(f'Theoretical excess S_tau^2/n:  {tau_i.var(ddof=1)/n:.4f}')
cover = np.mean(np.abs(est - true_ate) <= 1.96*np.sqrt(var_hat))
print(f'95% CI coverage: {cover:.3f}  (>= 0.95, as theory predicts)')

## Covariate adjustment: precision, not identification

In an RCT covariates are **not needed for identification** — but they reduce variance.
Freedman (2008) warned that plain OLS adjustment $Y \sim W + X$ can be *biased* in small
samples and can even hurt asymptotic precision with heterogeneous effects.
**Lin (2013)** showed the fix: regress with **treatment-demeaned-covariate interactions**

$$Y \sim W + \tilde{X} + W \cdot \tilde{X}, \qquad \tilde{X} = X - \bar{X}$$

The coefficient on $W$ is then asymptotically at least as efficient as the unadjusted
difference in means, whatever the true functional form. This is the ANCOVA estimator
recommended in Imbens & Rubin Ch. 7 and used by default in modern A/B testing platforms
(as the regression form of CUPED).


In [ ]:
# ── Lin (2013) interacted adjustment vs unadjusted vs naive ANCOVA ───────
def one_experiment(n=400):
    x  = np.random.normal(0, 1, n)
    y0 = 10 + 3*x + np.random.normal(0, 2, n)
    y1 = y0 + 5 + 4*x            # effect heterogeneity correlated with x
    w  = np.zeros(n, int); w[np.random.choice(n, n//2, replace=False)] = 1
    y  = np.where(w == 1, y1, y0)
    df = pd.DataFrame({'y': y, 'w': w, 'xt': x - x.mean()})

    est_dm    = y[w==1].mean() - y[w==0].mean()
    est_plain = smf.ols('y ~ w + xt', df).fit().params['w']
    est_lin   = smf.ols('y ~ w * xt', df).fit().params['w']
    return est_dm, est_plain, est_lin

res = np.array([one_experiment() for _ in range(3000)])
true_ate = 5.0
labels = ['Difference in means', 'OLS  y ~ w + x', 'Lin  y ~ w * (x - x_bar)']
print(f'{"estimator":<28} {"bias":>8} {"SD":>8} {"RMSE":>8}')
for j, lab in enumerate(labels):
    b = res[:, j].mean() - true_ate
    s = res[:, j].std()
    print(f'{lab:<28} {b:8.4f} {s:8.4f} {np.sqrt(b**2 + s**2):8.4f}')
print('\nAdjustment leaves the estimate unbiased and cuts the SD substantially;')
print('the interacted (Lin) version is never worse than either alternative.')

## Stratified (blocked) randomization

Randomizing *within strata* of a prognostic covariate guarantees balance on it by
construction (the design analogue of adjustment — see the Experimental Design course).
The stratified estimator is the weighted average of within-stratum differences:

$$\hat{\tau}_{strat} = \sum_g \frac{n_g}{n}\, \hat{\tau}_g, \qquad
\widehat{\text{Var}} = \sum_g \left(\frac{n_g}{n}\right)^2 \widehat{\text{Var}}(\hat{\tau}_g)$$

"Block what you can, randomize what you cannot" — Box, Hunter & Hunter.

## Key takeaways

| Concept | Statement |
|---|---|
| Fisher inference | Exact p-values from the randomization distribution; sharp null |
| Neyman inference | $\hat\tau = \bar Y_1 - \bar Y_0$ unbiased; variance estimate conservative |
| Freedman critique | Plain OLS adjustment can misbehave with heterogeneous effects |
| Lin estimator | Interact treatment with centered covariates — never hurts asymptotically |
| Stratification | Balance by design; analyze as designed (weight strata) |
| Gold standard | Randomization delivers identification *by design*, not by assumption |
